In [0]:
# now it is time to feature engineer...
spark.conf.set(
    "fs.azure.account.key.project60300347.dfs.core.windows.net",
    # github requires me to remove this part
)

In [0]:
df = spark.read.format("delta").load("abfss://lakehouse@project60300347.dfs.core.windows.net/curated/features_v1/")
df.printSchema()

root
 |-- sentiment: integer (nullable = true)
 |-- user: string (nullable = true)
 |-- date: date (nullable = true)
 |-- id: string (nullable = true)
 |-- query: string (nullable = true)
 |-- text: string (nullable = true)
 |-- total_tweets: long (nullable = true)
 |-- positive_tweets: long (nullable = true)
 |-- negative_tweets: long (nullable = true)
 |-- positive_ratio: double (nullable = true)
 |-- user_tweet_count: long (nullable = true)
 |-- sentiment_avg_length: double (nullable = true)
 |-- sentiment_tweet_count: long (nullable = true)



In [0]:
#  I will now follow the same data train, test, validation splits as what we performed in lab 4 
# creating the splits (70 train, 15 validation, 15 test)
train_df, val_df, test_df = df.randomSplit([0.7, 0.15, 0.15], seed=42)

In [0]:
# count check to make sure the split looks fine
print("Train rows:", train_df.count())
print("Validation rows:", val_df.count())
print("Test rows:", test_df.count())

Train rows: 1114480
Validation rows: 238219
Test rows: 238770


In [0]:
v2_path = "abfss://lakehouse@project60300347.dfs.core.windows.net/curated/features_v2/"

# putting each split as delta table
train_df.write.format("delta").mode("overwrite").save(v2_path + "train/")
val_df.write.format("delta").mode("overwrite").save(v2_path + "validation/")
test_df.write.format("delta").mode("overwrite").save(v2_path + "test/")

In [0]:
%pip install emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/608.4 kB ? eta -:--:--
   ━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/608.4 kB 932.4 kB/s eta 0:00:01
   ━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/608.4 kB 932.4 kB/s eta 0:00:01
   ━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.7/608.4 kB 638.4 kB/s eta 0:00:01
   ━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.2/608.4 kB 775.6 kB/s eta 0:00:01
   ━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.4/608.4 kB 824.9 kB/s eta 0:00:01
   ━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━ 235.5/608.4 kB 1.2 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━ 297.0/608.4 kB 1.2 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━ 522.2/608.4 kB 1.9 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 604.2/608.4 kB 1.9 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 1.8 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import nltk
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


True

In [0]:
# I will use the same exact goodreads_text_features.py code that I used in lab 4 along with the lexical bonus I added because why not... 
import os
from pyspark.sql import functions as F
from pyspark.sql.types import FloatType, StructType, StructField
from nltk.sentiment import SentimentIntensityAnalyzer
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF, IDFModel

# -------------------------------
# TEXT CLEANING
# -------------------------------
def clean_text(text):
    if text is None:
        return ""
    import re, emoji
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", " <URL> ", text)
    text = re.sub(r"\b\d+\b", " <NUM> ", text)
    text = emoji.replace_emoji(text, replace="<EMOJI>")
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# -------------------------------
# SENTIMENT ANALYZER
# -------------------------------
sia = SentimentIntensityAnalyzer()
def get_sentiment(text):
    if not text:
        return (0.0, 0.0, 0.0, 0.0)
    s = sia.polarity_scores(text)
    return (s["pos"], s["neu"], s["neg"], s["compound"])

# -------------------------------
# LEXICAL DIVERSITY
# -------------------------------
def lexical_diversity(text):
    if not text:
        return 0.0
    words = text.split()
    return len(set(words)) / len(words) if len(words) > 0 else 0.0

sentiment_schema = StructType([
    StructField("pos", FloatType()),
    StructField("neu", FloatType()),
    StructField("neg", FloatType()),
    StructField("compound", FloatType())
])

# -------------------------------
# MAIN PROCESS FUNCTION
# -------------------------------
def process_split(split_name, fit=False):
    print(f"\n--- Processing {split_name} split ---")

    # Load data from features_v1 (train/val/test created earlier)
    df = spark.read.format("delta").load(
        f"abfss://lakehouse@project60300347.dfs.core.windows.net/curated/features_v2/{split_name}/"
    )

    # Text cleaning
    from pyspark.sql.functions import udf
    clean_text_udf = udf(clean_text)
    df = df.withColumn("clean_text", clean_text_udf(F.col("text")))
    df = df.filter(F.length(F.col("clean_text")) >= 10)

    # Basic numeric text stats
    df = df.withColumn("tweet_length_words", F.size(F.split(F.col("clean_text"), " ")))
    df = df.withColumn("tweet_length_chars", F.length(F.col("clean_text")))

    # Lexical sentiment features (VADER)
    sentiment_udf = udf(get_sentiment, sentiment_schema)
    df = df.withColumn("sent", sentiment_udf(F.col("clean_text")))
    df = df.select("*",
        F.col("sent.pos").alias("lex_sent_pos"),
        F.col("sent.neu").alias("lex_sent_neu"),
        F.col("sent.neg").alias("lex_sent_neg"),
        F.col("sent.compound").alias("lex_sent_compound")
    ).drop("sent")

    # Lexical diversity
    lexdiv_udf = F.udf(lexical_diversity, FloatType())
    df = df.withColumn("lexical_diversity", lexdiv_udf(F.col("clean_text")))

    # -------------------------------
    # TF-IDF pipeline
    # -------------------------------
    tokenizer = Tokenizer(inputCol="clean_text", outputCol="words")
    df = tokenizer.transform(df)

    remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")
    df = remover.transform(df)

    hashing_tf = HashingTF(inputCol="filtered_words", outputCol="raw_features", numFeatures=300)
    df = hashing_tf.transform(df)

    idf = IDF(inputCol="raw_features", outputCol="tfidf_features")
    model_path = "/dbfs/tmp/tweets_idf_model"

    if fit:
        print("Fitting TF-IDF on training data...")
        idf_model = idf.fit(df)
        idf_model.write().overwrite().save(model_path)
    else:
        print("Loading existing TF-IDF model...")
        idf_model = IDFModel.load(model_path)

    df = idf_model.transform(df)

    # -------------------------------
    # Select final columns
    # -------------------------------
    final_cols = [
        "id", "user", "date", "query", "clean_text", "sentiment",
        "tweet_length_words", "tweet_length_chars",
        "lex_sent_pos", "lex_sent_neu", "lex_sent_neg", "lex_sent_compound",
        "lexical_diversity", "tfidf_features",
        "total_tweets", "positive_tweets", "negative_tweets", "positive_ratio",
        "user_tweet_count", "sentiment_avg_length", "sentiment_tweet_count"
    ]

    df_final = df.select(*[c for c in final_cols if c in df.columns])

    # -------------------------------
    # Write each split to its own Delta table
    # -------------------------------
    out_path = f"abfss://lakehouse@project60300347.dfs.core.windows.net/curated/features_v2_{split_name}/"
    df_final.write.format("delta").mode("overwrite").save(out_path)
    print(f"Saved {split_name} features to {out_path}")

# -------------------------------
# RUN PIPELINE FOR ALL SPLITS
# -------------------------------
process_split("train", fit=True)
process_split("validation", fit=False)
process_split("test", fit=False)


--- Processing train split ---
Fitting TF-IDF on training data...
Saved train features to abfss://lakehouse@project60300347.dfs.core.windows.net/curated/features_v2_train/

--- Processing validation split ---
Loading existing TF-IDF model...
Saved validation features to abfss://lakehouse@project60300347.dfs.core.windows.net/curated/features_v2_validation/

--- Processing test split ---
Loading existing TF-IDF model...
Saved test features to abfss://lakehouse@project60300347.dfs.core.windows.net/curated/features_v2_test/


In [0]:
# now i assign the train, test and validation to make checks on them
train_df = spark.read.format("delta").load(
  "abfss://lakehouse@project60300347.dfs.core.windows.net/curated/features_v2_train//"
)
val_df = spark.read.format("delta").load(
  "abfss://lakehouse@project60300347.dfs.core.windows.net/curated/features_v2_validation/"
)
test_df = spark.read.format("delta").load(
  "abfss://lakehouse@project60300347.dfs.core.windows.net/curated/features_v2_test/"
)

train_df.createOrReplaceTempView("final_train")
val_df.createOrReplaceTempView("final_validation")
test_df.createOrReplaceTempView("final_test")

In [0]:
# now ill just do checks to make sure that the data is valid and ready to be saved
# numeric feature summary
print("Numeric feature summary:")
train_df.select(
    "tweet_length_words",
    "tweet_length_chars",
    "lex_sent_compound",
    "lexical_diversity"
).summary().show()

# check tf-idf feature count
from pyspark.ml.linalg import VectorUDT
from pyspark.sql.functions import udf

tfidf_cols = [c for c in train_df.columns if c.startswith("tfidf_")]
print("TF-IDF feature columns found:", tfidf_cols)

if "tfidf_features" in train_df.columns:
    size_udf = udf(lambda v: int(v.size), "int")
    tfidf_size = train_df.select(size_udf("tfidf_features").alias("size")).limit(1).collect()[0]["size"]
    print(f"TF-IDF feature count: {tfidf_size}")
else:
    print("No TF-IDF feature column found.")

# null value check
print("\nNull value check per column:")
from pyspark.sql.functions import col, sum

null_counts = train_df.select(
    [sum(col(c).isNull().cast("int")).alias(c) for c in train_df.columns]
)
null_counts.show(truncate=False)

# total row count
print("\nTotal rows:", train_df.count())

# schema preview
print("\nSchema:")
train_df.printSchema()

Numeric feature summary:
+-------+------------------+------------------+------------------+-------------------+
|summary|tweet_length_words|tweet_length_chars| lex_sent_compound|  lexical_diversity|
+-------+------------------+------------------+------------------+-------------------+
|  count|           1094767|           1094767|           1094767|            1094767|
|   mean|  12.5900661967341| 62.82495910088631|0.1319499748190446| 0.9607000592517352|
| stddev| 6.703016073491831| 33.26368761767696|0.4464053617285189|0.06241325449787277|
|    min|                 1|                10|           -0.9985|        0.037037037|
|    25%|                 7|                35|           -0.1027|          0.9285714|
|    50%|                12|                58|               0.0|                1.0|
|    75%|                18|                89|            0.4939|                1.0|
|    max|                40|               177|            0.9987|                1.0|
+-------+---------